In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm

import copy

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import * # import NumericalFunctions 

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=4)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="LagrangianArrays", dataName="LFC",
                                dtype='float32')

In [ ]:
####################################
#CALCULATING FUNCTIONS

In [ ]:
#Getting Data
def GetData(t):
    #getting timestring for loading input data
    timeString = ModelData.timeStrings[t]

    #calculating variables
    varNames=['lfc','qv','th','prs']
    dataDictionary = {varName: CallVariable(ModelData, DataManager, timeString, varName) for varName in varNames}
    return dataDictionary

In [ ]:
#Calculating
def InitiateTimeArray():
    array = np.zeros((ModelData.Ntime))
    return array
def StoreMeans(data,t,array):
    mean = np.nanmean(data)
    array[t]=mean
    return array
def StoreMedians(data,t,array):
    mean = np.nanmedian(data)
    array[t]=mean
    return array
def StoreMins(data,t,array):
    coastX=int(ModelData.Nxh*0.25)
    data[data<=0]=np.nan
    min = np.nanmin(data[:,coastX:])
    array[t]=min
    return array
def StoreMaxes(data,t,array):
    max = np.nanmax(data)
    array[t]=max
    return array

In [ ]:
def GetAnalysisData(t):
    dataDictionary = GetData(t)
    LFC = dataDictionary['lfc']
    LFC[LFC < 0] = np.nan
    row, col = np.unravel_index(np.nanargmax(LFC), LFC.shape)
    
    qv = dataDictionary['qv']
    th = dataDictionary['th']
    p  = dataDictionary['prs']
    
    T    = th * (p / 100000) ** (2/7)
    Lv = 2.5e6; cp = 1005
    th_e = th * np.exp(Lv * qv / (cp * T))
    
    # Extract profiles at max LFC point
    qv_z   = qv[:, row, col]
    th_z   = th[:, row, col]    # was missing!
    th_e_z = th_e[:, row, col]
    
    return [LFC,qv,th,th_e, 
            qv_z,th_z,th_e_z]

In [ ]:
####################################
#PLOTTING FUNCTIONS

In [ ]:
def PlotTimeSeries(meanArrayLFC,medianArrayLFC,minArrayLFC,maxArrayLFC):
    times=np.arange(ModelData.Ntime)*ModelData.dt/3600+6
    
    fig,axes=plt.subplots(2,2,figsize=(10,10),sharey=True)
    axes=axes.ravel()
    axis=axes[0]
    axis.plot(times,meanArrayLFC)
    axis.set_xlabel('time (hrs)'); axis.set_ylabel('Mean LFC (m)')
    
    axis=axes[1]
    axis.plot(times,medianArrayLFC)
    axis.set_xlabel('time (hrs)'); axis.set_ylabel('Median LFC (m)')

    axis=axes[2]
    axis.plot(times,minArrayLFC)
    axis.set_xlabel('time (hrs)'); axis.set_ylabel('Min (right of coast) LFC (m)')
    
    axis=axes[3]
    axis.plot(times,maxArrayLFC)
    axis.set_xlabel('time (hrs)'); axis.set_ylabel('Max LFC (m)')

    axes[0].set_ylim(bottom=0)

In [ ]:
def PlotAnalysisData(LFC,qv,th,th_e,
                     qv_z,th_z,th_e_z,
                     t):
    
    fig = plt.figure(figsize=(12, 12))
    gs = gridspec.GridSpec(3, 3, figure=fig)
    
    # Top row: contourf spanning all 3 columns
    ax_map1 = fig.add_subplot(gs[0, :])  # row 0, all columns
    ax_map2 = fig.add_subplot(gs[1, :])  # row 1, all columns
    
    # Bottom row: 3 individual plots
    ax_qv = fig.add_subplot(gs[2, 0])
    ax_th = fig.add_subplot(gs[2, 1])
    ax_te = fig.add_subplot(gs[2, 2])
    

    #LFC contour
    cf1 = ax_map1.contourf(x, y, np.ma.masked_invalid(LFC), cmap='turbo',levels=np.arange(0,8e3+1,1e3),extend='max')
    fig.colorbar(cf1, ax=ax_map1,label='LFC (m)',pad=0)
    ax_map1.set_facecolor('grey')
    ax_map1.scatter(ModelData.xh[col], ModelData.yh[row], 
                   color='k', s=40, marker = 'x', zorder=5, label='maximum LFC')
    ax_map1.set_ylabel('y (km)');ax_map1.set_xlabel('x (km)')
    ax_map1.set_title(f'LFC at {times[t]} LT'); ax_map1.legend()
    
    #th_e contour
    cf2 = ax_map2.contourf(x, y, th_e[0], cmap='turbo',levels=9,extend='max')
    fig.colorbar(cf2, ax=ax_map2,label='θe (K)',pad=0)
    
    
    # qv profile
    ax_qv.plot(qv_z * 1e3, ModelData.zh, 'k')
    ax_qv.set_xlabel('qv (g/kg)'); ax_qv.set_ylabel('z (km)')
    ax_qv.set_ylim(0, 16); ax_qv.grid()
    
    # theta profile
    ax_th.plot(th_z, ModelData.zh, 'k')
    ax_th.set_xlabel('θ (K)')
    ax_th.set_ylim(0, 16); ax_th.grid()
    ax_th.set_title('Profile at max LFC location')
    
    # theta-e profile
    ax_te.plot(th_e_z, ModelData.zh, 'k')
    ax_te.set_xlabel('θe (K)')
    ax_te.set_ylim(0, 16); ax_te.grid()
    
    plt.tight_layout()

In [ ]:
###############################################################
#CALCULATING

In [ ]:
#CALCULATING AND APPENDING TO DATA EACH TIMESTEP
meanArrayLFC=InitiateTimeArray(); medianArrayLFC=InitiateTimeArray();
minArrayLFC=InitiateTimeArray(); maxArrayLFC=InitiateTimeArray();

loop_elements=range(ModelData.Ntime)
for t in tqdm(loop_elements, total=len(loop_elements)):
    dataDictionary = GetData(t)
    meanArrayLFC = StoreMeans(dataDictionary['lfc'],t,meanArrayLFC)
    medianArrayLFC = StoreMedians(dataDictionary['lfc'],t,medianArrayLFC)
    minArrayLFC = StoreMins(dataDictionary['lfc'],t,minArrayLFC)
    maxArrayLFC = StoreMaxes(dataDictionary['lfc'],t,maxArrayLFC)

In [ ]:
####################################
#PLOTTING

In [ ]:
PlotTimeSeries(meanArrayLFC,medianArrayLFC,minArrayLFC,maxArrayLFC)

In [ ]:
#Calculating
t=220
[LFC,qv,th,th_e,
 qv_z,th_z,th_e_z] = GetAnalysisData(t=t)

#Plotting
PlotAnalysisData(LFC,qv,th,th_e,
                 qv_z,th_z,th_e_z,
                 t=t)